# Evaluator-Optimizer Loop — Sprints

The executable test suite is authoritative. An evaluator can provide feedback, but refinement is bounded to three iterations to control latency and token use.

In [1]:
MAX_ITERATIONS = 3

def evaluator_optimizer(initial_code, tests, execute, evaluator, optimizer):
    """Run executable grading plus at most three evaluator-guided refinements.

    Curriculum:
        Module 13 (Evaluator-Optimizer and Reflection Patterns): bounded feedback loop.
        Module 2 (Python Functions, Testing and Debugging): executable tests are objective evidence.
    """
    code = initial_code
    history = []
    for iteration in range(MAX_ITERATIONS + 1):
        result = execute(code, tests)
        history.append({"iteration": iteration, "result": result})
        if result.get("status") != "OK" or result.get("all_passed"):
            break
        if iteration == MAX_ITERATIONS:
            break
        feedback = evaluator(code, result)
        code = optimizer(code, feedback)
    return {"final_code": code, "history": history, "iterations": len(history)-1, "cap": MAX_ITERATIONS}


In [2]:
# Deterministic cap check: a judge that never approves cannot cause an infinite loop.
calls = {"execute": 0, "evaluate": 0, "optimize": 0}
def fake_execute(code, tests):
    calls["execute"] += 1
    return {"status": "OK", "all_passed": False}
def fake_evaluator(code, result):
    calls["evaluate"] += 1
    return "still failing"
def fake_optimizer(code, feedback):
    calls["optimize"] += 1
    return code + "\n# refinement"
demo = evaluator_optimizer("def f(): return 0", [], fake_execute, fake_evaluator, fake_optimizer)
assert demo["iterations"] == 3
assert calls["execute"] == 4
assert calls["optimize"] == 3
demo

{'final_code': 'def f(): return 0\n# refinement\n# refinement\n# refinement',
 'history': [{'iteration': 0, 'result': {'status': 'OK', 'all_passed': False}},
  {'iteration': 1, 'result': {'status': 'OK', 'all_passed': False}},
  {'iteration': 2, 'result': {'status': 'OK', 'all_passed': False}},
  {'iteration': 3, 'result': {'status': 'OK', 'all_passed': False}}],
 'iterations': 3,
 'cap': 3}